In [14]:
from huggingface_hub import InferenceClient
from PIL import Image
import logging, os
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

In [28]:
class CoverGenerator:
    def __init__(self, model, token, log_path = "../logs/CoverGenerator.log", debug=False):
        dir = os.path.dirname(log_path)
        os.makedirs(dir, exist_ok=True)
        self.logger = logging.getLogger(f"{__name__}.CoverGenerator")
        self.logger.setLevel(logging.DEBUG if debug else logging.INFO)
        if self.logger.handlers:
            self.logger.handlers.clear()
        formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
        file_handler = logging.FileHandler(log_path, encoding='utf-8')
        file_handler.setLevel(logging.DEBUG if debug else logging.INFO)
        file_handler.setFormatter(formatter)
        self.logger.addHandler(file_handler)
        self.logger.propagate = False

        try:
            self.client = InferenceClient(model=model, token=token)
        except Exception as e:
            self.logger.error(f"An error occured during client making: {e}")
            raise
        
        self.logger.info("======================= New model =======================")
        self.logger.debug("Debug mode is ON")
        self.logger.info(f"CoverGenerator initialized with model=\'{model}\'")

    def generate(self, prompt, save_path):
        self.logger.info("Generating a new image.")
        self.logger.debug(f"Prompt: {prompt}")

        try:
            image = self.client.text_to_image(prompt)
        except Exception as e:
            self.logger.error(f"An error occured during generating {e}")
            raise
        
        try:
            dir = os.path.dirname(save_path)
            os.makedirs(dir, exist_ok=True)
            image.save(save_path)
        except Exception as e:
            self.logger.error(f"An error occured during saving image {e}")
        self.logger.info(f'Image saved to {save_path}')
            
        return image

In [29]:
BASE_DIR = Path(os.getcwd()).parent
load_dotenv(BASE_DIR / '.env')
TOKEN = os.getenv('HUGGINGFACE_TOKEN')

cover = CoverGenerator(
    model='black-forest-labs/FLUX.1-schnell',
    token=TOKEN,
    log_path="../logs/CoverGenerator.log",
    debug=True
)

In [30]:
data = pd.read_csv("../dataset/train1/prompts_train1.tsv", sep="\t")
id = data['id'][0]
prompt = data['prompt#0'][0]

In [31]:
img = cover.generate(
    prompt=prompt,
    save_path=f'../images/{id}.png'
)